In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 

In [2]:
titanic_data = pd.read_csv('./data/train.csv')

In [3]:
titanic_data.head(5)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
titanic_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [5]:
titanic_data.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [6]:
from sklearn.model_selection import train_test_split

X = titanic_data.drop(columns=['Survived', 'Cabin', 'PassengerId', 'Name', 'Ticket'])
y = titanic_data['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [7]:
X_train.shape , X_test.shape , y_train.shape , y_test.shape

((712, 7), (179, 7), (712,), (179,))

In [8]:
X.isna().sum()
#we have age some missing data in age column and in cabin also

Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [9]:
from sklearn.impute import SimpleImputer

age_imputer = SimpleImputer(strategy='median')
X_train[['Age']] = age_imputer.fit_transform(X_train[['Age']])  
X_test[['Age']]  = age_imputer.transform(X_test[['Age']])     

emb_imputer = SimpleImputer(strategy='most_frequent')
X_train[['Embarked']] = emb_imputer.fit_transform(X_train[['Embarked']])
X_test[['Embarked']]  = emb_imputer.transform(X_test[['Embarked']])

print(X_train.isna().sum())
print(X_test.isna().sum())

Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


In [10]:
X_test.info()

<class 'pandas.DataFrame'>
Index: 179 entries, 565 to 637
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    179 non-null    int64  
 1   Sex       179 non-null    str    
 2   Age       179 non-null    float64
 3   SibSp     179 non-null    int64  
 4   Parch     179 non-null    int64  
 5   Fare      179 non-null    float64
 6   Embarked  179 non-null    str    
dtypes: float64(2), int64(3), str(2)
memory usage: 11.2 KB


In [13]:
# we have sex and Embark column is not numerical so we have to make it numerical 
from sklearn.preprocessing import OneHotEncoder


X_train['Sex'] = X_train['Sex'].map({'male': 0, 'female': 1})
X_test['Sex']  = X_test['Sex'].map({'male': 0, 'female': 1})

emb_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
train_emb = emb_encoder.fit_transform(X_train[['Embarked']]) 
test_emb  = emb_encoder.transform(X_test[['Embarked']])       

emb_cols = emb_encoder.get_feature_names_out(['Embarked'])

train_emb_df = pd.DataFrame(train_emb, columns=emb_cols, index=X_train.index)
test_emb_df  = pd.DataFrame(test_emb,  columns=emb_cols, index=X_test.index)

X_train = pd.concat([X_train.drop(columns='Embarked'), train_emb_df], axis=1)
X_test  = pd.concat([X_test.drop(columns='Embarked'),  test_emb_df],  axis=1)

X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
692,3,0,28.5,0,0,56.4958,0.0,0.0,1.0
481,2,0,28.5,0,0,0.0000,0.0,0.0,1.0
527,1,0,28.5,0,0,221.7792,0.0,0.0,1.0
855,3,1,18.0,0,1,9.3500,0.0,0.0,1.0
801,2,1,31.0,1,1,26.2500,0.0,0.0,1.0


In [14]:
print(X_train.shape, X_test.shape)
print(X_train.dtypes)

(712, 9) (179, 9)
Pclass          int64
Sex             int64
Age           float64
SibSp           int64
Parch           int64
Fare          float64
Embarked_C    float64
Embarked_Q    float64
Embarked_S    float64
dtype: object


In [16]:

from sklearn.svm import LinearSVC


from sklearn.neighbors import KNeighborsClassifier


from sklearn.svm import SVC


from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   
X_test_scaled  = scaler.transform(X_test)  


In [17]:
models = {
    "LinearSVC": LinearSVC(),
    "KNN": KNeighborsClassifier(),
    "SVC": SVC(),
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForestClassifier": RandomForestClassifier(random_state=42)

}
results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    results[name] = {
        "train": model.score(X_train_scaled, y_train),
        "test":  model.score(X_test_scaled, y_test)
    }

pd.DataFrame(results).T.sort_values("test", ascending=False)

,train,test
KNN,0.851124,0.815642
SVC,0.844101,0.815642
RandomForestClassifier,0.983146,0.815642
LogisticRegression,0.806180,0.804469
LinearSVC,0.797753,0.793296
